### Notebook 01: Database — SQLite Schema and SQL Analysis

**CS 210 Final Project** | What the Headlines Say: Measuring Cognitive Bias and Agency Framing in AI News Discourse | Sadhana Vasanthakumar

Loads the 179,372-headline corpus into a normalized SQLite database and runs analytical SQL queries. The schema keeps raw scraped data separate from computed NLP features — this turned out to matter during PSI calibration, where the scoring formula was revised three times. Each revision dropped and recreated only `headlines_features`. With a flat table, those revisions would have risked touching the source data. That's the update anomaly protection 3NF actually provides in practice.

#### Schema — four tables in `bias_observatory.db`

| Table | Key | Description |
|---|---|---|
| `headlines_raw` | `id` (PK) | Immutable scraped corpus — never modified after loading |
| `headlines_features` | `headline_id` (FK) | NLP outputs from notebook 02 — scores, flags, bias dimensions |
| `publishers` | `publisher_id` (PK) | One row per outlet, `political_lean` joined from AllSides |
| `analysis_results` | `id` (PK) | Aggregated metrics from the SQL queries |

Foreign key: `headlines_features.headline_id` → `headlines_raw.id`
AllSides coverage: ~10.3% of headlines by volume (0.9% of unique publishers — the major outlets are over-represented).

#### Inputs

| File | Source | Required |
|------|--------|----------|
| `data/ai_headlines_full.csv` | Notebook 00 output | Yes |
| `data/allsides.csv` | Downloaded from AllSideR GitHub | Optional |

#### Outputs

| File | Purpose |
|------|---------|
| `data/bias_observatory.db` | Normalized SQLite database used by notebooks 02 and 03 |
| `outputs/allsides_coverage.json` | AllSides match coverage stats |

#### Dependencies

```bash
pip install pandas numpy
```

SQLite3 ships with Python. No other dependencies.

#### How to run

Takes about 5–8 minutes end-to-end. Cell 4 (inserting 179K rows) is the slowest at around 3 minutes.

1. Confirm `data/ai_headlines_full.csv` exists from notebook 00
2. Run all cells top to bottom
3. The database closes at the end of cell 8 — notebook 02 reopens it

In [1]:
# imports the four libraries needed by this notebook and defines database path
# creates data/ and outputs/ folders if they do not exist

import sqlite3
import pandas as pd
import numpy as np
import os
from datetime import datetime

DB_PATH = "data/bias_observatory.db"
RAW_CSV = "data/ai_headlines_full.csv"

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print(f"database target: {os.path.abspath(DB_PATH)}")
print(f"source csv:      {os.path.abspath(RAW_CSV)}")
print(f"run started:     {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

database target: c:\Users\Sadhana\Desktop\cs210_bias_project\data\bias_observatory.db
source csv:      c:\Users\Sadhana\Desktop\cs210_bias_project\data\ai_headlines_full.csv
run started:     2026-05-04 18:56:36


In [2]:
# Estimated runtime: 1 to 2 minutes (84 MB CSV read)

# load the full CSV and run integrity checks before touching the database.
# using ai_headlines_full.csv (not raw) because the analytical queries in
# cell 5 need category, window, query_source, and url — the raw file only
# has the 4 Figshare-compatible columns.

# integrity assertions before any database operations begin.
# if any assertion fails, the database is not created --
# this prevents loading corrupt data into SQLite.
#
# Why ai_headlines_full.csv and not ai_headlines_raw.csv:
#   The full file contains the category, window, query_source,
#   and url columns that the analytical queries in Cell 5 rely
#   on. The raw file only has the four Figshare-compatible
#   columns and would require a JOIN against the full file for
#   every category-level query.

df = pd.read_csv(RAW_CSV, encoding='utf-8', parse_dates=['date'])

print(f"loaded: {len(df):,} rows x {len(df.columns)} columns")
print(f"columns: {list(df.columns)}")
print(f"date range: {df['date'].min().date()} to {df['date'].max().date()}")
print()

assert df['title'].isna().sum() == 0,         "FAIL: missing titles found"
assert df['publisher'].isna().sum() == 0,     "FAIL: missing publishers found"
assert df['date'].isna().sum() == 0,          "FAIL: missing dates found"
assert df['title'].duplicated().sum() == 0,   "FAIL: duplicate titles found"
assert len(df) > 100_000,                     "FAIL: expected >100K rows from Notebook 00"

print("all integrity checks passed:")
print(f"   total headlines:   {len(df):,}")
print(f"   unique publishers: {df['publisher'].nunique():,}")
print(f"   unique categories: {df['category'].nunique()}")
print(f"   unique windows:    {df['window'].nunique()}")
print()
print("sample (first 3 rows):")
print(df[['headline_id','title','publisher','date','category','window']].head(3).to_string(index=False))

loaded: 179,372 rows x 11 columns
columns: ['headline_id', 'title', 'publisher', 'date', 'year', 'month', 'query_source', 'category', 'window', 'url', 'year_month']
date range: 2024-08-01 to 2026-03-31

all integrity checks passed:
   total headlines:   179,372
   unique publishers: 13,019
   unique categories: 18
   unique windows:    20

sample (first 3 rows):
 headline_id                                                                                                        title        publisher       date             category  window
           1                                                       Meta posts strong earnings amid growing AI investments Silicon Republic 2024-08-01 Companies & Products 2024-08
           2                                                          AI for ESG: Benefits, challenges and the CIO's role       TechTarget 2024-08-01          Environment 2024-08
           3 AI will transform even the humble PC, tech leaders say, as the hardware story moves b

In [3]:
# create the 4-table schema from scratch.
# foreign keys on so SQLite enforces referential integrity between
# headlines_raw and headlines_features.
# indexes on window, publisher, category, date — these are the columns
# every GROUP BY and WHERE clause in cell 5 touches.
#
# why 3NF instead of one flat table:
# political_lean is a property of the outlet, not the headline. storing
# it in headlines_raw would repeat the same value for every article from
# that publisher — that's exactly the redundancy 2NF prohibits.
# the JOIN cost (~150ms on 179K rows) is worth the isolation.

# Drops any existing database and recreates the four-table
# schema from scratch. Foreign key enforcement is enabled
# via PRAGMA. Indexes are created on the columns most
# frequently used in WHERE and GROUP BY clauses.
#
# trade-offs considered:
#   The JOIN between headlines_raw and headlines_features on
#   179K rows adds ~150ms vs a flat-table query. This is
#   acceptable given the isolation benefit. Indexes on the
#   foreign key column (headline_id is already the PK of
#   headlines_features) and on the GROUP BY columns (window,
#   publisher, category) keep aggregation queries fast.
#
# constraint decisions:
#   title TEXT NOT NULL: headlines are the analysis unit; null is meaningless
#   publisher TEXT NOT NULL: required for the political lean join
#   FOREIGN KEY (headline_id): referential integrity between raw and features
#   name TEXT UNIQUE in publishers: prevents duplicate rows on re-insert

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print('removed existing database for clean rebuild.')

conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON')
cur = conn.cursor()

DDL = 'CREATE TABLE headlines_raw (id INTEGER PRIMARY KEY, title TEXT NOT NULL, publisher TEXT NOT NULL, date TEXT NOT NULL, year INTEGER NOT NULL, month INTEGER NOT NULL, category TEXT, query_source TEXT, window TEXT, url TEXT);\nCREATE INDEX idx_raw_window    ON headlines_raw(window);\nCREATE INDEX idx_raw_publisher ON headlines_raw(publisher);\nCREATE INDEX idx_raw_category  ON headlines_raw(category);\nCREATE INDEX idx_raw_date      ON headlines_raw(date);\n\nCREATE TABLE headlines_features (headline_id INTEGER PRIMARY KEY, vader_compound REAL, vader_pos REAL, vader_neg REAL, vader_neu REAL, psi_flag TEXT, psi_score REAL, aai_score REAL, bias_fear REAL, bias_optimism REAL, bias_anthropomorphism REAL, bias_moral_panic REAL, bias_agency REAL, bias_techno_utopianism REAL, bias_economic_threat REAL, bias_control_loss REAL, bias_intensity REAL, bias_diversity INTEGER, dominant_bias TEXT, invisible_human_score REAL, FOREIGN KEY (headline_id) REFERENCES headlines_raw(id));\n\nCREATE TABLE publishers (publisher_id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE NOT NULL, political_lean TEXT, headline_count INTEGER);\nCREATE INDEX idx_pub_lean ON publishers(political_lean);\n\nCREATE TABLE analysis_results (id INTEGER PRIMARY KEY AUTOINCREMENT, metric TEXT NOT NULL, time_period TEXT, category TEXT, publisher TEXT, value REAL, n_headlines INTEGER);'

cur.executescript(DDL)
conn.commit()

tables = [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()]
indexes = [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='index' AND name NOT LIKE 'sqlite_%' ORDER BY name"
).fetchall()]

print('schema created.')
print(f'   tables:  {tables}')
print(f'   indexes: {indexes}')

removed existing database for clean rebuild.
schema created.
   tables:  ['analysis_results', 'headlines_features', 'headlines_raw', 'publishers', 'sqlite_sequence']
   indexes: ['idx_pub_lean', 'idx_raw_category', 'idx_raw_date', 'idx_raw_publisher', 'idx_raw_window']


In [4]:
# insert all 179K headlines into headlines_raw and populate the publishers
# table. chunksize=5000 keeps memory under 60MB — loading the whole thing
# in one transaction uses ~250MB during COMMIT and breaks on low-memory machines.
# political_lean left NULL here, cell 7 fills it in from AllSides.

df_insert = df[[
    'headline_id', 'title', 'publisher', 'date',
    'year', 'month', 'category', 'query_source', 'window', 'url'
]].copy()
df_insert['date'] = df_insert['date'].astype(str).str[:10]
df_insert = df_insert.rename(columns={'headline_id': 'id'})

df_insert.to_sql('headlines_raw', conn, if_exists='append',
                 index=False, chunksize=5000)

# one row per unique publisher with headline count
pub_counts = df['publisher'].value_counts().reset_index()
pub_counts.columns = ['name', 'headline_count']
pub_counts['political_lean'] = None
pub_counts.to_sql('publishers', conn, if_exists='append', index=False)

conn.commit()

n_raw = cur.execute("SELECT COUNT(*) FROM headlines_raw").fetchone()[0]
n_pub = cur.execute("SELECT COUNT(*) FROM publishers").fetchone()[0]
db_mb = os.path.getsize(DB_PATH) / (1024 * 1024)

print("insert complete:")
print(f"   headlines_raw:  {n_raw:,} rows")
print(f"   publishers:     {n_pub:,} rows")
print(f"   database size:  {db_mb:.1f} MB")

insert complete:
   headlines_raw:  179,372 rows
   publishers:     13,019 rows
   database size:  102.8 MB


In [5]:
# seven SQL queries, each answering a real research question.
# SQL features used across them:
#   Q1: window function SUM() OVER () for percentage-of-total
#   Q2: subquery in SELECT for relative percentage
#   Q3: HAVING clause to filter aggregated results
#   Q4: CASE inside SUM for time-period pivot
#   Q5: CAST AS REAL for accurate integer division
#   Q6: WHERE + GROUP BY together
#   Q7: CASE in SELECT for two-bucket grouping

def sql(label, query):
    print(f"\nQUERY: {label}")
    print('-' * 60)
    result = pd.read_sql_query(query, conn)
    print(result.to_string(index=False))
    return result


# Q1 -- monthly volume and publisher diversity
# window function: SUM(COUNT(*)) OVER () gives the total without a subquery
q1 = sql("monthly headline volume + publisher diversity", '''
    SELECT
        window,
        COUNT(*)                  AS n_headlines,
        COUNT(DISTINCT publisher) AS n_publishers,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_corpus
    FROM headlines_raw
    GROUP BY window
    ORDER BY window;
''')

# Q2 -- coverage by topic category
q2 = sql("coverage by topic category", '''
    SELECT
        category,
        COUNT(*)                  AS n_headlines,
        COUNT(DISTINCT publisher) AS n_publishers,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM headlines_raw), 2) AS pct
    FROM headlines_raw
    GROUP BY category
    ORDER BY n_headlines DESC;
''')

# Q3 -- concentration check: publishers with >0.5% of corpus
q3 = sql("publishers with > 0.5pct of corpus -- concentration check", '''
    SELECT
        publisher,
        COUNT(*) AS n,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM headlines_raw), 3) AS pct_of_corpus
    FROM headlines_raw
    GROUP BY publisher
    HAVING pct_of_corpus > 0.5
    ORDER BY n DESC;
''')

# Q4 -- category volume across four time periods (CASE inside SUM pivot)
q4 = sql("category volume: 4 time periods (h2 2024 vs h1 2025 vs h2 2025 vs q1 2026)", '''
    SELECT
        category,
        SUM(CASE WHEN window BETWEEN '2024-08' AND '2024-12' THEN 1 ELSE 0 END) AS h2_2024,
        SUM(CASE WHEN window BETWEEN '2025-01' AND '2025-06' THEN 1 ELSE 0 END) AS h1_2025,
        SUM(CASE WHEN window BETWEEN '2025-07' AND '2025-12' THEN 1 ELSE 0 END) AS h2_2025,
        SUM(CASE WHEN window BETWEEN '2026-01' AND '2026-03' THEN 1 ELSE 0 END) AS q1_2026
    FROM headlines_raw
    GROUP BY category
    ORDER BY (h1_2025 + h2_2025) DESC;
''')

# Q5 -- publisher diversity growth (CAST AS REAL for accurate division)
q5 = sql("publisher diversity growth by window", '''
    SELECT
        window,
        COUNT(DISTINCT publisher) AS unique_publishers,
        COUNT(*)                  AS headlines,
        ROUND(CAST(COUNT(*) AS REAL) / COUNT(DISTINCT publisher), 2) AS headlines_per_publisher
    FROM headlines_raw
    GROUP BY window
    ORDER BY window;
''')

# Q6 -- agency & autonomy by month — pre-NLP signal for PSI
q6 = sql("agency and autonomy category by month -- pre-PSI signal", '''
    SELECT
        window,
        COUNT(*) AS agency_headlines
    FROM headlines_raw
    WHERE category = 'Agency & Autonomy'
    GROUP BY window
    ORDER BY window;
''')

# Q7 -- human impact vs everything else — pre-NLP signal for Invisible Human
q7 = sql("human impact vs all other categories -- invisible human pre-NLP signal", '''
    SELECT
        CASE WHEN category = 'Society & Human Impact' THEN 'Human Impact'
             ELSE 'All Other Categories'
        END AS category_group,
        COUNT(*) AS n_headlines,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM headlines_raw), 2) AS pct
    FROM headlines_raw
    GROUP BY category_group
    ORDER BY n_headlines DESC;
''')

print("\nall queries complete.")


QUERY: monthly headline volume + publisher diversity
------------------------------------------------------------
 window  n_headlines  n_publishers  pct_of_corpus
2024-08         6500          2299           3.62
2024-09         6990          2426           3.90
2024-10         7453          2521           4.16
2024-11         6736          2444           3.76
2024-12         6849          2434           3.82
2025-01         7942          2605           4.43
2025-02         8303          2827           4.63
2025-03         8380          2735           4.67
2025-04         8643          2785           4.82
2025-05         9004          2867           5.02
2025-06         9165          2874           5.11
2025-07         9777          2963           5.45
2025-08         9548          2888           5.32
2025-09         9800          3037           5.46
2025-10        10339          3180           5.76
2025-11        10330          3266           5.76
2025-12        10303          3280 

In [6]:
# persist the query results to analysis_results so notebook 02 can read
# them without re-running the aggregations

records = []

for _, row in q1.iterrows():
    records.append({
        'metric': 'monthly_volume',
        'time_period': str(row['window']),
        'category': None,
        'publisher': None,
        'value': float(row['n_headlines']),
        'n_headlines': int(row['n_headlines'])
    })

for _, row in q2.iterrows():
    records.append({
        'metric': 'category_volume',
        'time_period': 'full_dataset',
        'category': str(row['category']),
        'publisher': None,
        'value': float(row['n_headlines']),
        'n_headlines': int(row['n_headlines'])
    })

for _, row in q5.iterrows():
    records.append({
        'metric': 'publisher_diversity',
        'time_period': str(row['window']),
        'category': None,
        'publisher': None,
        'value': float(row['unique_publishers']),
        'n_headlines': int(row['headlines'])
    })

pd.DataFrame(records).to_sql('analysis_results', conn,
                              if_exists='append', index=False)
conn.commit()

n = cur.execute("SELECT COUNT(*) FROM analysis_results").fetchone()[0]
print(f"saved {n} aggregate records to analysis_results")

saved 58 aggregate records to analysis_results


In [7]:
# load AllSides political lean ratings from the local CSV.
# originally tried fetching from the GitHub raw URL but it returns 404
# because the file was renamed upstream. loading from local file is
# more reliable for reproducibility anyway.
# file: data/allsides.csv — download from:
# https://raw.githubusercontent.com/favstats/AllSideR/master/data/allsides_data.csv

import pandas as pd
import os

ALLSIDES_PATH = "data/allsides.csv"

if not os.path.exists(ALLSIDES_PATH):
    raise FileNotFoundError(
        f"AllSides file not found at {ALLSIDES_PATH}. "
        f"Download from "
        f"https://raw.githubusercontent.com/favstats/AllSideR/master/data/allsides_data.csv "
        f"and save as data/allsides.csv"
    )

allsides = pd.read_csv(ALLSIDES_PATH)
print(f"loaded AllSides: {len(allsides):,} rated outlets")
print(f"   columns: {list(allsides.columns)}")
print()

# normalize publisher names for matching
allsides["news_source_norm"] = allsides["news_source"].str.lower().str.strip()

publishers = pd.read_sql_query(
    "SELECT DISTINCT publisher FROM headlines_raw", conn
)
publishers["publisher_norm"] = publishers["publisher"].str.lower().str.strip()

print(f"total unique publishers in our corpus: {len(publishers):,}")

matched = publishers.merge(
    allsides[["news_source_norm", "rating", "rating_num", "type"]],
    left_on="publisher_norm",
    right_on="news_source_norm",
    how="inner",
)

print(f"publishers matched to AllSides: {len(matched):,}")
print(f"match rate: {len(matched) / len(publishers) * 100:.2f}%")
print()

# how many headlines does the match cover?
if len(matched) > 0:
    matched_publishers_set = set(matched["publisher"])
    placeholders = ",".join("?" * len(matched_publishers_set))
    headline_match_count = pd.read_sql_query(
        f"SELECT COUNT(*) AS n FROM headlines_raw WHERE publisher IN ({placeholders})",
        conn,
        params=list(matched_publishers_set),
    ).iloc[0]["n"]
else:
    headline_match_count = 0

total_headlines = pd.read_sql_query("SELECT COUNT(*) AS n FROM headlines_raw", conn).iloc[0]["n"]
print(f"headlines covered by match: {headline_match_count:,} of {total_headlines:,} ({headline_match_count/total_headlines*100:.1f}%)")

print("\nlean distribution of matched publishers:")
print(matched["rating"].value_counts().to_string())

matched.to_sql("publishers_lean", conn, if_exists="replace", index=False)
conn.commit()
print("\nwrote publishers_lean table to SQLite")

loaded AllSides: 547 rated outlets
   columns: ['news_source', 'rating', 'rating_num', 'type', 'agree', 'disagree', 'perc_agree', 'url', 'editorial_review', 'blind_survey', 'third_party_analysis', 'independent_research', 'confidence_level', 'twitter', 'wiki', 'facebook', 'screen_name']

total unique publishers in our corpus: 13,019
publishers matched to AllSides: 118
match rate: 0.91%

headlines covered by match: 18,523 of 179,372 (10.3%)

lean distribution of matched publishers:
rating
center          39
left-center     35
left            21
right-center    16
right            7

wrote publishers_lean table to SQLite


In [8]:
# backfill political_lean into the publishers table for the 118 matched outlets.
# publishers table uses 'name'; publishers_lean uses 'publisher' — match on those.

import sqlite3
import pandas as pd

DB_PATH = "data/bias_observatory.db"

# reopen connection if closed
try:
    conn.execute("SELECT 1")
except (sqlite3.ProgrammingError, sqlite3.OperationalError, NameError):
    conn = sqlite3.connect(DB_PATH)

conn.execute("""
    UPDATE publishers
    SET political_lean = (
        SELECT pl.rating
        FROM publishers_lean pl
        WHERE pl.publisher = publishers.name
    )
    WHERE name IN (SELECT publisher FROM publishers_lean)
""")
conn.commit()

print("publishers table after backfill — distribution of political_lean:")
result = pd.read_sql_query("""
    SELECT
        COALESCE(political_lean, 'No AllSides Rating') AS lean,
        COUNT(*) AS n_publishers,
        SUM(headline_count) AS n_headlines
    FROM publishers
    GROUP BY lean
    ORDER BY n_publishers DESC
""", conn)
print(result.to_string(index=False))
print()

print("sample matched publishers:")
sample = pd.read_sql_query("""
    SELECT name, political_lean, headline_count
    FROM publishers
    WHERE political_lean IS NOT NULL
    ORDER BY headline_count DESC
    LIMIT 10
""", conn)
print(sample.to_string(index=False))
print()

print("sample JOIN: headlines x publishers, only matched (first 5):")
join_sample = pd.read_sql_query("""
    SELECT h.title, h.publisher, h.window, p.political_lean
    FROM headlines_raw h
    JOIN publishers p ON h.publisher = p.name
    WHERE p.political_lean IS NOT NULL
    LIMIT 5
""", conn)
print(join_sample.to_string(index=False))

publishers table after backfill — distribution of political_lean:
              lean  n_publishers  n_headlines
No AllSides Rating         12901       160849
            center            39        12545
       left-center            35         4044
              left            21          937
      right-center            16          588
             right             7          409

sample matched publishers:
            name political_lean  headline_count
          Forbes         center            2315
         Reuters         center            1910
            CNBC         center            1732
Business Insider         center            1686
      TechCrunch         center            1574
    The Guardian    left-center            1085
           Axios         center             911
       The Verge    left-center             557
            CNET         center             455
        Mashable           left             449

sample JOIN: headlines x publishers, only matched (firs

In [9]:
# AllSides coverage transparency — two percentages for the report.
# volume coverage (~10%) is much higher than publisher-count coverage (~0.9%)
# because AllSides covers the large mainstream outlets that produce the most
# headlines, while the long tail of niche publishers is unrated.
# any claim about political lean must cite these numbers and specify
# it applies to the ~10% sub-sample, not the full corpus.

print("AllSides coverage — transparency metrics for report")
print()

n_total_pubs   = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM publishers", conn).iloc[0]["n"]
n_matched_pubs = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM publishers WHERE political_lean IS NOT NULL",
    conn).iloc[0]["n"]
pub_coverage_pct = n_matched_pubs / n_total_pubs * 100

n_total_rows = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM headlines_raw", conn).iloc[0]["n"]
n_matched_rows = pd.read_sql_query("""
    SELECT COUNT(*) AS n
    FROM headlines_raw r
    JOIN publishers p ON r.publisher = p.name
    WHERE p.political_lean IS NOT NULL
""", conn).iloc[0]["n"]
vol_coverage_pct = n_matched_rows / n_total_rows * 100

print(f"   publisher-count coverage:")
print(f"      matched publishers:   {n_matched_pubs:,} / {n_total_pubs:,}")
print(f"      coverage rate:        {pub_coverage_pct:.2f}%")

print(f"\n   headline-volume coverage:")
print(f"      matched headlines:    {n_matched_rows:,} / {n_total_rows:,}")
print(f"      coverage rate:        {vol_coverage_pct:.2f}%")

print()
print(f"   AllSides covers the major mainstream outlets, which is why")
print(f"   {vol_coverage_pct:.1f}% of headlines have lean ratings even though only")
print(f"   {pub_coverage_pct:.1f}% of unique publishers do. any claim about political")
print(f"   lean refers to this {vol_coverage_pct:.0f}% sub-sample, not the full corpus.")

import json
coverage_record = {
    "publishers_total":       int(n_total_pubs),
    "publishers_matched":     int(n_matched_pubs),
    "publisher_coverage_pct": round(float(pub_coverage_pct), 2),
    "headlines_total":        int(n_total_rows),
    "headlines_matched":      int(n_matched_rows),
    "volume_coverage_pct":    round(float(vol_coverage_pct), 2),
}
with open("outputs/allsides_coverage.json", "w") as f:
    json.dump(coverage_record, f, indent=2)
print(f"\nsaved: outputs/allsides_coverage.json")

AllSides coverage — transparency metrics for report

   publisher-count coverage:
      matched publishers:   118 / 13,019
      coverage rate:        0.91%

   headline-volume coverage:
      matched headlines:    18,523 / 179,372
      coverage rate:        10.33%

   AllSides covers the major mainstream outlets, which is why
   10.3% of headlines have lean ratings even though only
   0.9% of unique publishers do. any claim about political
   lean refers to this 10% sub-sample, not the full corpus.

saved: outputs/allsides_coverage.json


In [10]:
# final state check and close. headlines_features will be 0 rows here —
# that's expected, notebook 02 populates it.

print("final database state:")
for table in ['headlines_raw', 'headlines_features', 'publishers', 'analysis_results']:
    n = cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"   {table:<25} {n:>8,} rows")

db_mb = os.path.getsize(DB_PATH) / (1024 * 1024)
print(f"   database size:            {db_mb:.1f} MB")

print()
print("sample JOIN query (headlines_raw x publishers, only matched):")
sample = pd.read_sql_query('''
    SELECT
        r.title,
        r.publisher,
        r.window,
        p.political_lean
    FROM headlines_raw r
    LEFT JOIN publishers p ON r.publisher = p.name
    WHERE p.political_lean IS NOT NULL
    LIMIT 8;
''', conn)
print(sample.to_string(index=False))

print()
print("lean distribution at headline level (JOIN aggregation):")
lean_dist = pd.read_sql_query('''
    SELECT
        COALESCE(p.political_lean, 'No AllSides Rating') AS lean,
        COUNT(*) AS n_headlines,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM headlines_raw), 2) AS pct
    FROM headlines_raw r
    LEFT JOIN publishers p ON r.publisher = p.name
    GROUP BY lean
    ORDER BY n_headlines DESC;
''', conn)
print(lean_dist.to_string(index=False))

conn.close()
print()
print("database closed. next: run 02_nlp_pipeline.ipynb")

final database state:
   headlines_raw              179,372 rows
   headlines_features               0 rows
   publishers                  13,019 rows
   analysis_results                58 rows
   database size:            102.9 MB

sample JOIN query (headlines_raw x publishers, only matched):
                                                                                title        publisher  window political_lean
                Taco Bell to expand AI-powered drive-thrus as rivals ditch technology    New York Post 2024-08          right
                                Indian Billionaire Birla May Have a Problem of Plenty        Bloomberg 2024-08         center
                                         Media groups seek a new profit model with AI  Financial Times 2024-08         center
Meta scraps failed celebrity AI chatbots after users ignored them, deemed them creepy    New York Post 2024-08          right
                        Remember the Meta AI celebrity avatars? They're get